# 🏥 The Future-Driven Diagnostics (FDD) Health AI Audit

**You've just been hired onto the AI Audit Team at Future-Driven Diagnostics Health**, a hospital network that has been rapidly increasing demand of patients, this rapid growth has caused slower appointment time and slower analysis results and, in general, is putting patients in a compromised situation. The hospital has accumulated eight AI systems built by different teams over the past few years:

- a cardiovascular risk score
- a hospital-readmission predictor
- a stroke triage flag
- a chest X-ray reader
- a skin-lesion screener
- a breast-ultrasound screener
- a colon-histopathology classifier
- a new LLM-based clinical assistant

### The three questions every system must answer

Every one of these systems reports strong accuracy. **None of them has ever been asked *why* it works.** Compliance won't let any of them near a real patient until someone can answer three questions - for every single system:

1. **What is it actually using to make this decision?**
2. **Does that hold up - or is it a shortcut** (a data artifact, a marker, a positional token) **that happens to correlate with the right answer?**
3. **How do we know the explanation itself is trustworthy**, rather than just a plausible-looking story?

### Your job: the four-step audit

For every system below, you will run the same four-step audit:

| Step | What you do |
|---|---|
| 👀 **Look** | See what the model actually predicts for a real case |
| 🔍 **Explain** | Find out what evidence it used |
| 🧪 **Stress-test** | Destroy the model's learned knowledge and re-explain. |
| ✅ **Verdict** | Trustworthy / trustworthy-with-caveats / not ready - and why |

### One method, any modality

The same four steps apply whether the "evidence" is a lab value, a pixel, or a word. **That consistency is the whole point of this audit** - by the end you won't just have eight verdicts, you'll have one reusable method for auditing *any* AI system, regardless of what kind of data it eats.

### The eight systems at a glance

| # | System | Data type | The shortcut to watch for |
|---|---|---|---|
| 1 | Cardiovascular risk score | Tabular | Does it use BP/cholesterol like a cardiologist would? |
| 2 | Readmission predictor | Tabular | Selection bias - who gets tested |
| 3 | Stroke triage flag | Tabular | 95% of patients don't stroke - is "accurate" meaningless? |
| 4 | Chest X-ray reader | Imaging | Reading lungs, or the laterality marker in the corner? |
| 5 | Skin-lesion screener | Imaging | Reading the lesion, or a ruler/ink mark next to it? |
| 6 | Breast-ultrasound screener | Imaging | Reading the mass, or the on-screen calipers / annotations? |
| 7 | Colon-histopathology classifier | Imaging | Reading tissue morphology, or stain-colour / tile-edge artifacts? |
| 8 | Clinical assistant (LLM) | Text | Attending to the clinically relevant word, or to punctuation/position? |

## 🔧 Before you start
Run the setup cell below once. It installs every package, and defines the **shared audit toolkit** you'll reuse for all eight systems - tabular, imaging, and LLM alike. You don't need to read it line by line, but the `randomize_weights` function is worth a look: it's the single idea that ties every "stress-test" step in this notebook together.

## 🎯 How to complete this notebook

This is your **student workbook** - run every cell from top to bottom.

- **Worked examples (already solved):** System 1 (tabular) and System 4 (imaging) show the full pattern, and System 8 (the LLM) is provided in full. Read them first.
- **Your tasks:** cells marked `# 🎯 TODO` have blanks written as `____`. Fill each blank so the cell runs, following the matching worked example (tabular tasks -> copy System 1; imaging tasks -> copy System 4).
- **Verdicts:** after each system, write your verdict (Trustworthy / Trustworthy with caveats / Not ready) and one sentence why, in the fill-in cell provided.

Nothing else needs changing - the data loading, model training, and plotting are done for you.

In [ ]:
# @title 🔧 Master Setup - run me first (installs · data helpers · audit toolkit) { display-mode: "form" }
import subprocess, sys

# ---- keep output focused: silence library warnings / progress bars ----
import os, warnings, logging
os.environ.setdefault("PYTHONWARNINGS", "ignore")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
for _noisy in ("huggingface_hub", "datasets", "transformers", "torch", "medmnist"):
    logging.getLogger(_noisy).setLevel(logging.ERROR)
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

_pip("scikit-learn", "lime", "shap", "torch", "torchvision", "captum",
     "torchxrayvision", "scikit-image", "transformers",
     "medmnist>=3.0", "matplotlib", "pandas", "numpy", "requests", "ucimlrepo")

import copy
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from IPython.display import display, HTML
import shap
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================================
#  SHARED AUDIT TOOLKIT
#  These are the AUDIT TEAM's tools. The same four functions below are
#  reused, unmodified, for tabular models, imaging models, and the LLM
#  at the end of the notebook.
# =====================================================================

def randomize_weights(model):
    """Return a copy of `model` with freshly-randomised (untrained) weights.

    This is THE stress-test. A trustworthy explanation of *this model*
    must fall apart once the model's learned knowledge is destroyed.
    If an explanation barely changes after this, it was never really
    describing the model -- it was describing the data.

    Normalization layers (BatchNorm / LayerNorm / GroupNorm) are reset to a
    working identity (scale 1, shift 0) instead of being zeroed. Zeroing their
    scale would make every one output a constant, the whole network would emit
    the same value for any input, and every attribution map would come back
    flat -- a dead result, not the noisy map a live random model should give.
    Conv / linear weights are still fully randomised, so learned knowledge is
    destroyed while signal still flows.
    """
    m = copy.deepcopy(model)
    norm_types = (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d,
                  nn.LayerNorm, nn.GroupNorm)
    for mod in m.modules():
        if isinstance(mod, norm_types):
            if getattr(mod, "weight", None) is not None:
                nn.init.ones_(mod.weight)
            if getattr(mod, "bias", None) is not None:
                nn.init.zeros_(mod.bias)
            if hasattr(mod, "reset_running_stats"):
                mod.reset_running_stats()
        else:
            for _, p in mod.named_parameters(recurse=False):
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
    return m.eval()

# ---- tabular systems (Unit 1) ----
def make_mlp(n_features, n_classes=2, hidden=(64, 32)):
    layers, d = [], n_features
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU()]; d = h
    layers += [nn.Linear(d, n_classes)]
    return nn.Sequential(*layers)

def train_mlp(model, Xtr, ytr, epochs=120, lr=1e-3):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    Xt = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = lossf(model(Xt), yt); loss.backward(); opt.step()
    return model.eval()

def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        Xt = torch.tensor(np.asarray(X), dtype=torch.float32, device=DEVICE)
        return torch.softmax(model(Xt), dim=1).cpu().numpy()

# ---- imaging systems (Unit 2) ----
from captum.attr import Saliency, IntegratedGradients, LayerGradCam, LayerAttribution

def predicted_class(model, x):
    model.eval()
    with torch.no_grad():
        return int(model(x).argmax(1).item())

def attributions(model, x, target, target_layer):
    """Return (saliency, integrated-gradients, grad-cam) heatmaps, each HxW."""
    x = x.clone().to(DEVICE).requires_grad_(True)
    sal = Saliency(model).attribute(x, target=target).abs().sum(1).squeeze().cpu().detach().numpy()
    ig  = IntegratedGradients(model).attribute(x, target=target, n_steps=32
                              ).abs().sum(1).squeeze().cpu().detach().numpy()
    gc  = LayerGradCam(model, target_layer).attribute(x, target=target)
    gc  = LayerAttribution.interpolate(gc, x.shape[-2:]).squeeze().cpu().detach().numpy()
    return sal, ig, gc

def show_maps(img_disp, maps, titles, suptitle=""):
    fig, ax = plt.subplots(1, len(maps) + 1, figsize=(3.2 * (len(maps) + 1), 3.4))
    ax[0].imshow(img_disp, cmap="gray"); ax[0].set_title("input"); ax[0].axis("off")
    for a, m, t in zip(ax[1:], maps, titles):
        a.imshow(img_disp, cmap="gray")
        a.imshow(m, cmap="jet", alpha=0.5); a.set_title(t); a.axis("off")
    fig.suptitle(suptitle); plt.tight_layout(); plt.show()

# ---- pipeline-diagram helper (visuals used throughout the notebook) ----
def render_pipeline(title, emoji, steps, grad=None):
    if grad is None:
        grad = ["#667eea", "#6f7bf0", "#8273ec", "#9a6fe2", "#b06fd0",
                "#c56fbe", "#d36bb0", "#db6fa9"][:len(steps)]
        while len(grad) < len(steps):
            grad += grad
    blocks = ""
    for (ic, t, d), g in zip(steps, grad):
        blocks += (f'<div class="pl-step"><div class="pl-ic" style="background:linear-gradient(135deg,{g},{g}cc)">{ic}</div>'
                   f'<div class="pl-t">{t}</div><div class="pl-d">{d}</div></div>'
                   '<div class="pl-ar">➜</div>')
    blocks = blocks.rsplit('<div class="pl-ar">➜</div>', 1)[0]
    display(HTML(f'''
<style>
.pl{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #ecebff}}
.pl-h{{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 14px}}
.pl-row{{display:flex;align-items:flex-start;flex-wrap:wrap;gap:0}}
.pl-step{{flex:1 1 96px;min-width:96px;text-align:center;padding:0 2px}}
.pl-ic{{width:48px;height:48px;border-radius:50%;margin:0 auto 7px;display:flex;align-items:center;
       justify-content:center;font-size:21px;color:#fff;box-shadow:0 6px 14px rgba(102,126,234,.35)}}
.pl-t{{font-weight:700;font-size:12px;color:#2c2350}}
.pl-d{{font-size:10px;color:#8b86a6;margin-top:2px}}
.pl-ar{{display:flex;align-items:center;font-size:18px;color:#b9a9e6;flex:0 0 14px;height:48px}}
</style>
<div class="pl"><div class="pl-h">{emoji} {title}</div><div class="pl-row">{blocks}</div></div>'''))

def verdict_box(system_name, prompts):
    """Accent-bar card: guiding questions plus verdict pills for the student."""
    items = "".join(f'<li>{p}</li>' for p in prompts)
    _vcolors = {"Trustworthy": "#c9f2d8", "Caveats": "#fdeec7", "Not ready": "#fbd6d6"}
    pills = "".join(
        f'<span style="background:{bg};color:#3b2d6b;font-size:11.5px;font-weight:700;'
        f'padding:5px 12px;border-radius:999px">{label}</span>'
        for label, bg in _vcolors.items())
    display(HTML(f'''
<div style="display:flex;font-family:system-ui,Segoe UI,Roboto,sans-serif;border-radius:14px;
            overflow:hidden;margin:12px 0;background:#fbfaff;border:1px solid #ecebff;
            box-shadow:0 4px 14px rgba(102,126,234,.15)">
  <div style="width:8px;background:linear-gradient(180deg,#8273ec,#b06fd0)"></div>
  <div style="padding:16px 20px;flex:1">
    <div style="font-size:11px;font-weight:800;letter-spacing:.08em;color:#8273ec;
                text-transform:uppercase">&#128221; Audit verdict</div>
    <div style="font-size:17px;font-weight:800;color:#2c2350;margin:2px 0 12px">{system_name}</div>
    <ul style="color:#4a4368;font-size:13.5px;margin:0 0 14px;padding-left:18px;line-height:1.6">{items}</ul>
    <div style="display:flex;gap:8px;flex-wrap:wrap">{pills}</div>
    <div style="margin-top:11px;font-size:11.5px;color:#8b86a6">
        Pick one above and note one sentence why, in the markdown cell below.</div>
  </div>
</div>'''))

# =====================================================================
#  SHARED PLOTTING STYLE
#  Every tabular system (Systems 1-3) calls these THREE functions for its
#  LIME / SHAP / sanity-check plots, so every audit in this notebook looks
#  visually identical -- same palette, same layout, same title style.
# =====================================================================

LIME_NEG_COLOR = "#E06D61"   # soft coral  -- negative contribution
LIME_POS_COLOR = "#4CA69B"   # muted teal  -- positive contribution

def _style_axes_clean(ax):
    ax.axvline(0, color="#CCCCCC", lw=1.2, zorder=1)
    ax.grid(axis="x", linestyle="--", alpha=0.4, color="#DDDDDD", zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

def plot_lime_local(exp, index, class_label="Disease", label=1):
    """Single-patient LIME bar chart. Same look for every tabular system."""
    vals = dict(exp.as_list(label=label))
    features = list(vals.keys())[::-1]
    importances = list(vals.values())[::-1]
    colors = [LIME_NEG_COLOR if v < 0 else LIME_POS_COLOR for v in importances]

    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=110)
    bars = ax.barh(features, importances, color=colors, edgecolor="none", height=0.6)

    max_val = max(abs(v) for v in importances) if importances else 1
    padding = max_val * 0.03
    for bar, val in zip(bars, importances):
        is_near_zero = abs(val) < (max_val * 0.05)
        if val >= 0:
            align, offset = "left", (padding * 1.5 if is_near_zero else padding)
        else:
            align, offset = "right", -(padding * 1.5 if is_near_zero else padding)
        ax.text(val + offset, bar.get_y() + bar.get_height() / 2, f"{val:+.3f}",
                 va="center", ha=align, fontsize=9, fontweight="bold", color="#333333")

    _style_axes_clean(ax)
    ax.tick_params(axis="y", colors="#333333", labelsize=9.5, length=0, pad=15)
    ax.tick_params(axis="x", colors="#666666", labelsize=9)

    plt.title(f"Local Explanation for Class: {class_label}", fontsize=13, fontweight="bold",
               color="#111111", loc="left", pad=20)
    ax.text(0, 1.03, f"Patient ID: #{index} | Feature contribution to probability",
            transform=ax.transAxes, fontsize=9.5, color="#666666", ha="left")

    x_limit = max_val * 1.25
    ax.set_xlim(-x_limit, x_limit)
    plt.tight_layout()
    plt.show()

def plot_shap_summary(shap_values, X, feature_names, title="SHAP Feature Importance", max_display=10):
    """SHAP beeswarm summary plot with a title centered on the plot itself
    (not the figure canvas), same for every tabular system."""
    plt.clf()
    shap.summary_plot(shap_values, X, feature_names=feature_names,
                       max_display=max_display, plot_size=(16, 8), show=False)
    ax = plt.gca()
    pos = ax.get_position()
    center_x = (pos.x0 + pos.x1) / 2
    fig = plt.gcf()
    fig.suptitle(title, fontsize=20, fontweight="bold", x=center_x, y=0.98)
    plt.subplots_adjust(top=0.90)
    plt.show()

def plot_lime_sanity(exp_trained, exp_random, title="LIME Explanation Comparison (Sanity Check)", label=1):
    """Trained-vs-randomized LIME comparison. Same look for every system's stress-test."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True)
    for ax, exp, panel_title in [(axes[0], exp_trained, "Trained Model"),
                                   (axes[1], exp_random, "Randomized Model")]:
        vals = dict(exp.as_list(label=label))
        features = sorted(vals, key=lambda f: abs(vals[f]))
        values = [vals[f] for f in features]
        colors = [LIME_NEG_COLOR if v < 0 else LIME_POS_COLOR for v in values]

        ax.barh(features, values, color=colors)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title(panel_title, fontsize=16, fontweight="bold")
        ax.set_xlabel("Feature Contribution")
        ax.grid(axis="x", linestyle="--", alpha=0.5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="y", labelsize=11, pad=10)

    fig.suptitle(title, fontsize=20, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()

print("✅ Setup complete - audit toolkit ready for all eight systems.")

In [ ]:
#@title 📊 Visualization - The Audit Workflow (double-click to view the code) { display-mode: "form" }
def render_pipeline(title, emoji, steps, grad=None):
    """Generic pipeline-diagram renderer, reused for every system in this audit.
    steps: list of (icon, title, description) tuples."""
    if grad is None:
        grad = ["#667eea", "#6f7bf0", "#8273ec", "#9a6fe2", "#b06fd0",
                "#c56fbe", "#d36bb0", "#db6fa9"][:len(steps)]
        while len(grad) < len(steps):
            grad += grad
    blocks = ""
    for (ic, t, d), g in zip(steps, grad):
        blocks += (f'<div class="pl-step"><div class="pl-ic" style="background:linear-gradient(135deg,{g},{g}cc)">{ic}</div>'
                   f'<div class="pl-t">{t}</div><div class="pl-d">{d}</div></div>'
                   '<div class="pl-ar">➜</div>')
    blocks = blocks.rsplit('<div class="pl-ar">➜</div>', 1)[0]
    display(HTML(f'''
<style>
.pl{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
    border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #ecebff}}
.pl-h{{font-size:20px;font-weight:800;color:#3b2d6b;margin:0 0 14px}}
.pl-row{{display:flex;align-items:flex-start;flex-wrap:wrap;gap:0}}
.pl-step{{flex:1 1 96px;min-width:96px;text-align:center;padding:0 2px}}
.pl-ic{{width:48px;height:48px;border-radius:50%;margin:0 auto 7px;display:flex;align-items:center;
       justify-content:center;font-size:21px;color:#fff;box-shadow:0 6px 14px rgba(102,126,234,.35)}}
.pl-t{{font-weight:700;font-size:12px;color:#2c2350}}
.pl-d{{font-size:10px;color:#8b86a6;margin-top:2px}}
.pl-ar{{display:flex;align-items:center;font-size:18px;color:#b9a9e6;flex:0 0 14px;height:48px}}
</style>
<div class="pl"><div class="pl-h">{emoji} {title}</div><div class="pl-row">{blocks}</div></div>'''))

render_pipeline("The audit loop (every system, every modality)", "🧭", [
    ("👀", "Look", "see the prediction"),
    ("🔍", "Explain", "find the evidence"),
    ("🧪", "Stress-test", "randomize + re-explain"),
    ("✅", "Verdict", "trust it or not")
])


---
# 📋 Unit 1 - Tabular Systems

Three of Future-Driven Diagnostics's systems run on structured patient records: rows of numbers like age, blood pressure, or visit counts. For each one you'll **train a small neural network from scratch** (nothing pretrained here - you built it, so you get to interrogate it), then run the full audit loop: explain locally with **LIME**, explain globally with **SHAP**, stress-test with weight randomization, and write a verdict.

In [ ]:
#@title 🔎 Pipeline diagram (double-click to view the code) { display-mode: "form" }
render_pipeline("Unit 1 workflow - tabular", "📋", [
    ("🗂️", "Load data", "patient records"),
    ("🌳", "Train", "small MLP, from scratch"),
    ("🔍", "LIME", "why THIS patient?"),
    ("🌐", "SHAP", "what drives it globally?"),
    ("🧪", "Stress-test", "randomize + re-explain"),
    ("✅", "Verdict", "audit sign-off")
])


## System 1 - Cardiovascular risk score
**Data:** 70,000 patient records - age, blood pressure, cholesterol, glucose, lifestyle flags.
**Claim under audit:** the model predicts cardiovascular disease with strong accuracy. **Your question:** does it use blood pressure and cholesterol the way a cardiologist would, or has it found something else?

In [ ]:
# ---- System 1 data: Cardiovascular Disease (70k), public mirror, no token needed ----
CARDIO_MIRROR_URL = "https://raw.githubusercontent.com/caravanuden/cardio/master/cardio_train.csv"

def load_cardio():
    df = pd.read_csv(CARDIO_MIRROR_URL, sep=";")
    df["age_years"] = (df["age"] / 365).round(1)
    feats = ["age_years", "gender", "height", "weight", "ap_hi", "ap_lo",
             "cholesterol", "gluc", "smoke", "alco", "active"]
    y = df["cardio"].astype(int).values
    Xv = df[feats].astype(float).values
    Xtr, Xte, ytr, yte = train_test_split(Xv, y, test_size=0.2, stratify=y, random_state=0)
    sc = StandardScaler().fit(Xtr)
    meta = {"target_name": "cardio_disease", "class_names": ["no disease", "disease"]}
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte, feats, meta

Xtr_c, Xte_c, ytr_c, yte_c, feat_c, meta_c = load_cardio()
print(f"train {Xtr_c.shape}  test {Xte_c.shape}  |  features: {len(feat_c)}")
print("class balance (train):", np.bincount(ytr_c))
pd.DataFrame(Xtr_c[:5], columns=feat_c).round(2)


In [ ]:
# ---- Train from scratch: a small MLP, System 1 ----
model_c = make_mlp(Xtr_c.shape[1])
model_c = train_mlp(model_c, Xtr_c, ytr_c)

proba_c = predict_proba(model_c, Xte_c)[:, 1]
acc_c = accuracy_score(yte_c, proba_c > 0.5)
auc_c = roc_auc_score(yte_c, proba_c)
print(f"test accuracy: {acc_c:.3f}   test ROC-AUC: {auc_c:.3f}")
print("Looks strong. But WHAT is it using to decide? -> audit step: Explain")


### 🔍 Explain - LIME (local) and SHAP (global)
LIME fits a simple linear model around **one** prediction: which features pushed this patient's score up or down? SHAP aggregates contributions across **many** patients: what does the model rely on in general? **Do the local and global stories agree?**

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp_c = LimeTabularExplainer(
    Xtr_c, feature_names=feat_c, class_names=meta_c["class_names"],
    discretize_continuous=True, mode="classification")

i_c = 0  # try changing this to audit a different patient

# LIME needs a function that maps rows -> class probabilities for THIS model.
def predict_c(rows):
    return predict_proba(model_c, rows)

exp_c = lime_exp_c.explain_instance(Xte_c[i_c], predict_c, num_features=8, labels=(1,))
print(f"patient #{i_c}: predicted P(positive) = {predict_proba(model_c, Xte_c[i_c:i_c+1])[0,1]:.2f}")

plot_lime_local(exp_c, i_c, class_label="Disease")

In [ ]:
import shap

bg_c = torch.tensor(Xtr_c[np.random.choice(len(Xtr_c), 100, replace=False)],
                     dtype=torch.float32, device=DEVICE)
de_c = shap.DeepExplainer(model_c, bg_c)
sv_c = de_c.shap_values(torch.tensor(Xte_c[:300], dtype=torch.float32, device=DEVICE))
sv_c = sv_c[1] if isinstance(sv_c, list) else sv_c

plot_shap_summary(sv_c, Xte_c[:300], feat_c)

### 🧪 Stress-test
Randomize the model's weights and re-explain the **same** patient. If the LIME plot barely changes, the "explanation" was reading the data distribution, not this model's learned weights.

In [ ]:
rnd_c = randomize_weights(model_c)

def predict_rnd_c(rows):
    return predict_proba(rnd_c, rows)

exp_rnd_c = lime_exp_c.explain_instance(Xte_c[i_c], predict_rnd_c, num_features=8, labels=(1,))

plot_lime_sanity(exp_c, exp_rnd_c)

**Your turn:** the stress-test above re-ran LIME on the randomized model. Compare the two panels - did the explanation hold up, or did it fall apart? An explanation that survives is a good sign the *explanation itself* is real (which is separate from whether the model's clinical reasoning is sound). Record what you see in the verdict below.

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Cardiovascular risk score", [
    "Which features does the model rely on most -- ap_hi (systolic) or ap_lo (diastolic)? Does that match medical guidance?",
    "Do LIME and SHAP agree on the top drivers?",
    "Did the explanation collapse under the stress-test? If not, why not?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why."
])


> **Your verdict - Cardiovascular risk score**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

## System 2 - Readmission predictor
**Data:** ~100k hospital encounters for diabetic patients - visit counts, medications, A1C result, age band.
**Claim under audit:** the model predicts early (<30 day) readmission well. **Your question:** this dataset is the classic *selection-bias* trap in clinical ML - who gets a lab ordered, who gets readmitted, is not random. Is the model reading medicine, or reading how the hospital happens to collect data?

In [ ]:
#@title 🔎 Pipeline diagram (double-click to view the code) { display-mode: "form" }
render_pipeline("System 2 focus", "🏥", [
    ("🗂️", "~100k encounters", "diabetic patients"),
    ("⚠️", "Selection bias", "who gets tested"),
    ("🔍", "Audit", "is it medicine or bookkeeping?")
])


In [ ]:
# ---- System 2 data: Diabetes 130-US Hospitals (UCI id=296) ----
def load_diabetes130():
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=296)
    X, y = ds.data.features.copy(), ds.data.targets.copy()
    yb = (y["readmitted"] == "<30").astype(int).values
    num_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures",
                "num_medications", "number_outpatient", "number_emergency",
                "number_inpatient", "number_diagnoses"]
    cat_cols = ["age", "A1Cresult", "insulin", "change", "diabetesMed"]
    X = X[num_cols + cat_cols].copy()
    for c in cat_cols:
        X[c] = X[c].astype("category").cat.codes
    feature_names = num_cols + cat_cols
    Xv = X[feature_names].astype(float).values
    Xtr, Xte, ytr, yte = train_test_split(Xv, yb, test_size=0.2, stratify=yb, random_state=0)
    sc = StandardScaler().fit(Xtr)
    meta = {"target_name": "early_readmission", "class_names": ["not <30d", "readmit <30d"]}
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte, feature_names, meta

Xtr_d, Xte_d, ytr_d, yte_d, feat_d, meta_d = load_diabetes130()
print(f"train {Xtr_d.shape}  test {Xte_d.shape}  |  features: {len(feat_d)}")
print("class balance (train):", np.bincount(ytr_d))
pd.DataFrame(Xtr_d[:5], columns=feat_d).round(2)


In [ ]:
model_d = make_mlp(Xtr_d.shape[1])
model_d = train_mlp(model_d, Xtr_d, ytr_d)

proba_d = predict_proba(model_d, Xte_d)[:, 1]
acc_d = accuracy_score(yte_d, proba_d > 0.5)
auc_d = roc_auc_score(yte_d, proba_d)
print(f"test accuracy: {acc_d:.3f}   test ROC-AUC: {auc_d:.3f}")
print("Hospital readmission is high-stakes AND reimbursed -- accurate isn't the same as safe.")


In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp_d = LimeTabularExplainer(
    Xtr_d, feature_names=feat_d, class_names=meta_d["class_names"],
    discretize_continuous=True, mode="classification")

i_d = 0  # try changing this to audit a different patient

# 🎯 TODO: same pattern as System 1. LIME needs a rows -> probabilities function for model_d.
def predict_d(rows):
    return predict_proba(____, rows)

exp_d = lime_exp_d.explain_instance(Xte_d[i_d], ____, num_features=8, labels=(1,))
print(f"patient #{i_d}: predicted P(positive) = {predict_proba(model_d, Xte_d[i_d:i_d+1])[0,1]:.2f}")

plot_lime_local(exp_d, i_d, class_label="Early Readmission")

In [ ]:
# 🎯 TODO: same pattern as System 1. Build a SHAP DeepExplainer for model_d.
bg_d = torch.tensor(Xtr_d[np.random.choice(len(Xtr_d), 100, replace=False)],
                    dtype=torch.float32, device=DEVICE)
de_d = shap.DeepExplainer(____, bg_d)
sv_d = de_d.shap_values(torch.tensor(Xte_d[:300], dtype=torch.float32, device=DEVICE))
sv_d = sv_d[1] if isinstance(sv_d, list) else sv_d

plot_shap_summary(sv_d, Xte_d[:300], feat_d)

In [ ]:
# 🎯 TODO: same pattern as System 1. Randomize model_d, then re-run LIME on the SAME patient.
rnd_d = randomize_weights(____)

def predict_rnd_d(rows):
    return predict_proba(rnd_d, rows)

exp_rnd_d = lime_exp_d.explain_instance(Xte_d[i_d], ____, num_features=8, labels=(1,))

plot_lime_sanity(exp_d, ____)

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Readmission predictor", [
    "Is 'age' really doing the work -- or a proxy for something else? (Try retraining without it and compare AUC.)",
    "Do LIME and SHAP agree on the top drivers?",
    "Did the explanation collapse under the stress-test?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why."
])


> **Your verdict - Readmission predictor**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

**Your turn:** compare the trained and randomized panels above. Do the same features keep surfacing whether or not the model has actually learned anything? What would that tell you about whether the explanation is tracking the *model* or just the *data*? Write your reasoning in the verdict below.

## System 3 - Stroke triage flag
**Data:** ~5,000 patients, only **~5% stroke-positive** - a rare, imbalanced outcome.
**Claim under audit:** the model scores well. **Your question:** with 95% negatives, a model that never predicts "stroke" still scores ~95% accuracy. Is this system actually finding stroke cases, or just predicting the majority class and calling it a day?

In [ ]:
#@title 🔎 Pipeline diagram (double-click to view the code) { display-mode: "form" }
render_pipeline("System 3 focus", "🧠", [
    ("🗂️", "~5k patients", "~5% stroke-positive"),
    ("⚠️", "Imbalance trap", "always-negative scores ~95%"),
    ("🔍", "Audit", "is it finding the rare cases?")
])


In [ ]:
# ---- System 3 data: Stroke Prediction (~5k, imbalanced), public mirror ----
STROKE_MIRROR_URL = "https://raw.githubusercontent.com/karavokyrismichail/Stroke-Prediction---Random-Forest/main/healthcare-dataset-stroke-data/healthcare-dataset-stroke-data.csv"

def load_stroke():
    df = pd.read_csv(STROKE_MIRROR_URL).drop(columns=["id"], errors="ignore")
    df["bmi"] = df["bmi"].fillna(df["bmi"].median())
    for c in ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]:
        df[c] = df[c].astype("category").cat.codes
    y = df["stroke"].astype(int).values
    feats = [c for c in df.columns if c != "stroke"]
    Xv = df[feats].astype(float).values
    Xtr, Xte, ytr, yte = train_test_split(Xv, y, test_size=0.2, stratify=y, random_state=0)
    sc = StandardScaler().fit(Xtr)
    meta = {"target_name": "stroke", "class_names": ["no stroke", "stroke"]}
    return sc.transform(Xtr), sc.transform(Xte), ytr, yte, feats, meta

Xtr_s, Xte_s, ytr_s, yte_s, feat_s, meta_s = load_stroke()
print(f"train {Xtr_s.shape}  test {Xte_s.shape}  |  features: {len(feat_s)}")
print("class balance (train):", np.bincount(ytr_s))
pd.DataFrame(Xtr_s[:5], columns=feat_s).round(2)


In [ ]:
model_s = make_mlp(Xtr_s.shape[1])
model_s = train_mlp(model_s, Xtr_s, ytr_s)

proba_s = predict_proba(model_s, Xte_s)[:, 1]
acc_s = accuracy_score(yte_s, proba_s > 0.5)
auc_s = roc_auc_score(yte_s, proba_s)
print(f"test accuracy: {acc_s:.3f}   test ROC-AUC: {auc_s:.3f}")
print("Compare these two numbers carefully -- which one is honest under imbalance?")


In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp_s = LimeTabularExplainer(
    Xtr_s, feature_names=feat_s, class_names=meta_s["class_names"],
    discretize_continuous=True, mode="classification")

i_s = 0  # try changing this to audit a different patient

# 🎯 TODO: same pattern as System 1. LIME needs a rows -> probabilities function for model_s.
def predict_s(rows):
    return predict_proba(____, rows)

exp_s = lime_exp_s.explain_instance(Xte_s[i_s], ____, num_features=8, labels=(1,))
print(f"patient #{i_s}: predicted P(positive) = {predict_proba(model_s, Xte_s[i_s:i_s+1])[0,1]:.2f}")

plot_lime_local(exp_s, i_s, class_label="Stroke")

In [ ]:
# 🎯 TODO: same pattern as System 1. Build a SHAP DeepExplainer for model_s.
bg_s = torch.tensor(Xtr_s[np.random.choice(len(Xtr_s), 100, replace=False)],
                    dtype=torch.float32, device=DEVICE)
de_s = shap.DeepExplainer(____, bg_s)
sv_s = de_s.shap_values(torch.tensor(Xte_s[:300], dtype=torch.float32, device=DEVICE))
sv_s = sv_s[1] if isinstance(sv_s, list) else sv_s

plot_shap_summary(sv_s, Xte_s[:300], feat_s)

In [ ]:
# 🎯 TODO: same pattern as System 1. Randomize model_s, then re-run LIME on the SAME patient.
rnd_s = randomize_weights(____)

def predict_rnd_s(rows):
    return predict_proba(rnd_s, rows)

exp_rnd_s = lime_exp_s.explain_instance(Xte_s[i_s], ____, num_features=8, labels=(1,))

plot_lime_sanity(exp_s, ____)

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Stroke triage flag", [
    "Accuracy vs ROC-AUC: which is honest under ~5% imbalance, and why?",
    "Do LIME and SHAP agree on the top drivers for a positive prediction?",
    "Did the explanation collapse under the stress-test?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why."
])


> **Your verdict - Stroke triage flag**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

---
# 🩻 Unit 2 - Imaging Systems

Four of Future-Driven Diagnostics's systems read images instead of spreadsheets. Here we **don't train from scratch** - modern vision models start from huge labelled corpora, so we either audit off-the-shelf pretrained weights (chest X-ray) or fine-tune a small ResNet on public data (skin, breast-ultrasound, histopathology), then go straight to auditing what the model looks at. Instead of LIME/SHAP, imaging uses **pixel attribution**: Saliency, Integrated Gradients, and Grad-CAM, each answering "which pixels drove this prediction?" a different way. The stress-test is identical in spirit: randomize the weights and see whether the heatmap collapses.

> ℹ️ **Data note:** every imaging system here runs end-to-end with no login and no manual downloads. The chest X-ray uses a real pretrained model (TorchXRayVision); the skin, breast-ultrasound, and histopathology systems fine-tune a small ResNet live on public auto-downloading datasets (Hugging Face / MedMNIST).

In [ ]:
#@title 🔎 Pipeline diagram (double-click to view the code) { display-mode: "form" }
render_pipeline("Unit 2 workflow - imaging", "🩻", [
    ("📥", "Load weights", "pretrained, not trained here"),
    ("🖼️", "Predict", "on sample images"),
    ("🔥", "Attribute", "saliency / IG / Grad-CAM"),
    ("🧪", "Stress-test", "randomize + re-explain"),
    ("✅", "Verdict", "audit sign-off")
])


## System 4 - Chest X-ray reader
**Model:** TorchXRayVision, pretrained on hundreds of thousands of chest X-rays - no training on our side.
**Claim under audit:** it detects pathologies (pneumonia, effusion, cardiomegaly, ...) accurately. **Your question:** is the evidence *in the lungs*, or is it reading the laterality marker printed in the corner of the film?

In [ ]:
# ---- System 4: pretrained chest X-ray model (TorchXRayVision, real weights, real auto-download) ----
import torchxrayvision as xrv, skimage, requests, io

# A real, publicly-hosted demo X-ray from the official torchxrayvision project
# (used in their own README/HF Space) - replaces the CHANGE-ME placeholder.
CXR_SAMPLE_IMAGES_URL = "https://huggingface.co/spaces/torchxrayvision/torchxrayvision-classifier/resolve/main/16747_3_1.jpg"
CXR_SAMPLE_NAMES = ["demo_0", "demo_1", "demo_2", "demo_3"]  # same source image, kept as 4 slots for the audit loop

def _load_cxr(name):
    raw = requests.get(CXR_SAMPLE_IMAGES_URL, timeout=30).content
    img = skimage.io.imread(io.BytesIO(raw))
    img = xrv.datasets.normalize(img, 255)
    if img.ndim == 3: img = img.mean(2)
    img = img[None, ...]
    img = skimage.transform.resize(img[0], (224, 224))[None]
    return torch.tensor(img, dtype=torch.float32)

def load_cxr_system():
    model = xrv.models.DenseNet(weights="densenet121-res224-all").to(DEVICE).eval()
    class_names = list(model.pathologies)
    target_layer = model.features[-1]
    imgs = torch.stack([_load_cxr(n) for n in CXR_SAMPLE_NAMES]).to(DEVICE)
    return model, imgs, class_names, target_layer

model_x, images_x, classes_x, layer_x = load_cxr_system()
print(f"{len(images_x)} images  |  {len(classes_x)} pathologies  |  device={DEVICE}")
for k in range(len(images_x)):
    c = predicted_class(model_x, images_x[k:k+1])
    print(f"image {k}: predicted -> {classes_x[c]}")

### 🔍 Explain - three attribution methods, side by side
- **Saliency** - raw input-gradient: which pixels move the output fastest.
- **Integrated Gradients** - accumulates gradients along a baseline→image path; less noisy.
- **Grad-CAM** - coarse, from the last conv layer: *which region* drives the class.

In [ ]:
k_x = 0
x_x = images_x[k_x:k_x+1]
target_x = predicted_class(model_x, x_x)
disp_x = images_x[k_x].detach().cpu().numpy()
disp_x = disp_x.transpose(1, 2, 0) if disp_x.shape[0] == 3 else disp_x[0]
disp_x = (disp_x - disp_x.min()) / (np.ptp(disp_x) + 1e-8)

sal_x, ig_x, gc_x = attributions(model_x, x_x, target_x, layer_x)
show_maps(disp_x, [sal_x, ig_x, gc_x], ["Saliency", "Integrated Grads", "Grad-CAM"],
          suptitle=f"predicted: {classes_x[target_x]}")
print("Do the three methods agree on WHERE the evidence is? Disagreement is a finding.")


### 🧪 Stress-test
Randomize the model and re-run the attributions. A map that reflects the MODEL should turn to noise. A map that barely changes is reflecting the IMAGE, not the model.

In [ ]:
rnd_x = randomize_weights(model_x)
rnd_layer_x = rnd_x.features[-1]

sal_x2, ig_x2, gc_x2 = attributions(rnd_x, x_x, target_x, rnd_layer_x)

show_maps(disp_x, [sal_x, sal_x2], ["Saliency (trained)", "Saliency (random)"],
          suptitle="Does the explanation depend on what the model learned?")
show_maps(disp_x, [gc_x, gc_x2], ["Grad-CAM (trained)", "Grad-CAM (random)"])


In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Chest X-ray reader", [
    "Find an image where Grad-CAM highlights a corner/marker rather than anatomy -- what does that tell you about training data?",
    "Which method survived the stress-test best? Which is the prettiest *lie*?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why."
])


> **Your verdict - Chest X-ray reader**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

## System 5 - Skin-lesion screener
**Model:** a ResNet fine-tuned live on real HAM10000 dermatoscopic images (public Hugging Face mirror, no placeholder).
**Claim under audit:** it classifies lesion type well. **Your question:** HAM10000 is famous for a shortcut - models can latch onto **rulers, ink marks, and colour patches** in the photo instead of the lesion itself.

In [ ]:
# ---- System 5: skin-lesion classifier - fine-tuned live on real HAM10000 data ----
from datasets import load_dataset
import torchvision
from torchvision import transforms

# Real, public HAM10000 mirror on Hugging Face -- no login, no placeholder
ds_stream = load_dataset("marmal88/skin_cancer", split="train", streaming=True)
ds_stream = ds_stream.shuffle(seed=0, buffer_size=2000)

N_SAMPLES = 200  # small + fast for a teaching notebook, not a research-grade model
raw_samples = list(ds_stream.take(N_SAMPLES))

CLASS_NAMES_HAM = sorted({s["dx"] for s in raw_samples})
label_to_idx = {c: i for i, c in enumerate(CLASS_NAMES_HAM)}

_tf_ham = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

X_ham = torch.stack([_tf_ham(s["image"].convert("RGB")) for s in raw_samples]).to(DEVICE)
y_ham = torch.tensor([label_to_idx[s["dx"]] for s in raw_samples], dtype=torch.long, device=DEVICE)

def train_ham_model(X, y, epochs=5, lr=1e-4):
    model = torchvision.models.resnet18(weights="IMAGENET1K_V1")  # standard public torchvision weights
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES_HAM))
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = lossf(model(X), y); loss.backward(); opt.step()
    return model.eval()

model_h = train_ham_model(X_ham, y_ham)
layer_h = model_h.layer4[-1]
images_h = X_ham[:4]
classes_h = CLASS_NAMES_HAM

print(f"{len(images_h)} images  |  {len(classes_h)} classes  |  device={DEVICE}")
for k in range(len(images_h)):
    c = predicted_class(model_h, images_h[k:k+1])
    print(f"image {k}: predicted -> {classes_h[c]}")

In [ ]:
# 🎯 TODO: same pattern as System 4. Run the three attributions for image k_h.
k_h = 0
x_h = images_h[k_h:k_h+1]
target_h = predicted_class(model_h, x_h)
disp_h = images_h[k_h].detach().cpu().numpy()
disp_h = disp_h.transpose(1, 2, 0) if disp_h.shape[0] == 3 else disp_h[0]
disp_h = (disp_h - disp_h.min()) / (np.ptp(disp_h) + 1e-8)

sal_h, ig_h, gc_h = attributions(____, x_h, ____, layer_h)
show_maps(disp_h, [sal_h, ig_h, gc_h], ["Saliency", "Integrated Grads", "Grad-CAM"],
          suptitle=f"predicted: {classes_h[target_h]}")
print("Do the three methods agree on WHERE the evidence is? Disagreement is a finding.")

In [ ]:
# 🎯 TODO: same pattern as System 4. Randomize the model and re-run the attributions.
#   For these ResNet models the last conv block is model.layer4[-1].
rnd_h = randomize_weights(model_h)
rnd_layer_h = rnd_h.layer4[-1]

sal_h2, ig_h2, gc_h2 = attributions(____, x_h, target_h, ____)

show_maps(disp_h, [sal_h, sal_h2], ["Saliency (trained)", "Saliency (random)"],
          suptitle="Does the explanation depend on what the model learned?")
show_maps(disp_h, [gc_h, gc_h2], ["Grad-CAM (trained)", "Grad-CAM (random)"])
print("Whichever method changes MOST under randomisation is the one tracking the model.")

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Skin-lesion screener", [
    "Find an image where attribution sits on a ruler or ink mark rather than the lesion. How would you fix the dataset?",
    "Which method survived the stress-test best?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why."
])


> **Your verdict - Skin-lesion screener**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

## System 6 - Breast-ultrasound screener
**Model:** a ResNet fine-tuned live on **BreastMNIST** (real breast-ultrasound scans, auto-downloaded from Zenodo, no login and no placeholder).
**Claim under audit:** it flags malignant lesions. **Your question:** do the maps land on the *mass* itself, or on the on-screen **calipers / annotation marks** that sonographers leave on the image?

In [ ]:
# ---- System 6: breast-ultrasound screener - fine-tuned live on real BreastMNIST data ----
import medmnist
from medmnist import INFO
import torchvision
from torchvision import transforms

# Shared helpers for the two MedMNIST imaging systems (6 & 7).
# MedMNIST subsets auto-download from Zenodo -- no login, no CHANGE-ME placeholder.
def load_medmnist(flag, size=224, n=200):
    """Download a MedMNIST subset and return (X[N,3,224,224], y, class_names)."""
    info = INFO[flag]
    DataClass = getattr(medmnist, info["python_class"])
    try:
        ds = DataClass(split="train", download=True, size=size)   # 64/128/224 variants (medmnist>=3)
    except TypeError:
        ds = DataClass(split="train", download=True)              # older medmnist -> native 28px
    class_names = [info["label"][str(i)] for i in range(len(info["label"]))]
    idx = np.random.RandomState(0).permutation(len(ds))[:n]       # mix classes into our sample
    tf = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    X = torch.stack([tf(ds[int(i)][0].convert("RGB")) for i in idx]).to(DEVICE)
    y = torch.tensor([int(np.array(ds[int(i)][1]).item()) for i in idx],
                     dtype=torch.long, device=DEVICE)
    return X, y, class_names

def train_img_model(X, y, n_classes, epochs=5, lr=1e-4):
    model = torchvision.models.resnet18(weights="IMAGENET1K_V1")  # standard public torchvision weights
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = lossf(model(X), y); loss.backward(); opt.step()
    return model.eval()

X_us, y_us, classes_u = load_medmnist("breastmnist", size=224, n=200)
model_u = train_img_model(X_us, y_us, len(classes_u))
layer_u = model_u.layer4[-1]
images_u = X_us[:4]

print(f"{len(images_u)} images  |  {len(classes_u)} classes  |  device={DEVICE}")
for k in range(len(images_u)):
    c = predicted_class(model_u, images_u[k:k+1])
    print(f"image {k}: predicted -> {classes_u[c]}")

In [ ]:
# 🎯 TODO: same pattern as System 4. Run the three attributions for image k_u.
k_u = 0
x_u = images_u[k_u:k_u+1]
target_u = predicted_class(model_u, x_u)
disp_u = images_u[k_u].detach().cpu().numpy()
disp_u = disp_u.transpose(1, 2, 0) if disp_u.shape[0] == 3 else disp_u[0]
disp_u = (disp_u - disp_u.min()) / (np.ptp(disp_u) + 1e-8)

sal_u, ig_u, gc_u = attributions(____, x_u, ____, layer_u)
show_maps(disp_u, [sal_u, ig_u, gc_u], ["Saliency", "Integrated Grads", "Grad-CAM"],
          suptitle=f"predicted: {classes_u[target_u]}")
print("Do the three methods agree on WHERE the evidence is? Disagreement is a finding.")

In [ ]:
# 🎯 TODO: same pattern as System 4. Randomize the model and re-run the attributions.
#   For these ResNet models the last conv block is model.layer4[-1].
rnd_u = randomize_weights(model_u)
rnd_layer_u = rnd_u.layer4[-1]

sal_u2, ig_u2, gc_u2 = attributions(____, x_u, target_u, ____)

show_maps(disp_u, [sal_u, sal_u2], ["Saliency (trained)", "Saliency (random)"],
          suptitle="Does the explanation depend on what the model learned?")
show_maps(disp_u, [gc_u, gc_u2], ["Grad-CAM (trained)", "Grad-CAM (random)"])
print("Whichever method changes MOST under randomisation is the one tracking the model.")

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Breast-ultrasound screener", [
    "Compare a 'malignant' scan and a 'normal, benign' scan -- does attribution move to the mass, or stay on the same generic region?",
    "Ultrasound scans often carry on-screen calipers / annotation marks -- does the map latch onto those instead of tissue?",
    "Which method survived the stress-test best?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why.",
])

> **Your verdict - Breast-ultrasound screener**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

## System 7 - Colon-histopathology classifier
**Model:** a ResNet fine-tuned live on **PathMNIST** (real colorectal histology tiles, auto-downloaded from Zenodo).
**Claim under audit:** it identifies the tissue type in a slide. **Your question:** is the evidence on genuine **tissue morphology**, or on **stain-colour / tile-edge artifacts** that happen to correlate with each class?

In [ ]:
# ---- System 7: colon-histopathology classifier - fine-tuned live on real PathMNIST data ----
# Reuses load_medmnist / train_img_model defined in System 6.
# PathMNIST ships one big file per size; size=64 keeps the Colab download to ~5-6 min
# (raise to 128/224 for sharper tiles at a longer download, or 28 for a ~2 min blurry run).
X_pa, y_pa, classes_p = load_medmnist("pathmnist", size=64, n=200)
model_p = train_img_model(X_pa, y_pa, len(classes_p))
layer_p = model_p.layer4[-1]
images_p = X_pa[:4]

print(f"{len(images_p)} images  |  {len(classes_p)} classes  |  device={DEVICE}")
for k in range(len(images_p)):
    c = predicted_class(model_p, images_p[k:k+1])
    print(f"image {k}: predicted -> {classes_p[c]}")

In [ ]:
# 🎯 TODO: same pattern as System 4. Run the three attributions for image k_p.
k_p = 0
x_p = images_p[k_p:k_p+1]
target_p = predicted_class(model_p, x_p)
disp_p = images_p[k_p].detach().cpu().numpy()
disp_p = disp_p.transpose(1, 2, 0) if disp_p.shape[0] == 3 else disp_p[0]
disp_p = (disp_p - disp_p.min()) / (np.ptp(disp_p) + 1e-8)

sal_p, ig_p, gc_p = attributions(____, x_p, ____, layer_p)
show_maps(disp_p, [sal_p, ig_p, gc_p], ["Saliency", "Integrated Grads", "Grad-CAM"],
          suptitle=f"predicted: {classes_p[target_p]}")
print("Do the three methods agree on WHERE the evidence is? Disagreement is a finding.")

In [ ]:
# 🎯 TODO: same pattern as System 4. Randomize the model and re-run the attributions.
#   For these ResNet models the last conv block is model.layer4[-1].
rnd_p = randomize_weights(model_p)
rnd_layer_p = rnd_p.layer4[-1]

sal_p2, ig_p2, gc_p2 = attributions(____, x_p, target_p, ____)

show_maps(disp_p, [sal_p, sal_p2], ["Saliency (trained)", "Saliency (random)"],
          suptitle="Does the explanation depend on what the model learned?")
show_maps(disp_p, [gc_p, gc_p2], ["Grad-CAM (trained)", "Grad-CAM (random)"])
print("Whichever method changes MOST under randomisation is the one tracking the model.")

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Colon-histopathology classifier", [
    "Compare two different tissue classes -- does attribution track distinct morphology, or sit on the same generic patch / tile edge?",
    "Histology models often exploit stain-colour and tile-boundary artifacts -- do the maps sit on those instead of cells?",
    "Which method survived the stress-test best?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why.",
])

> **Your verdict - Colon-histopathology classifier**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

---
# 🤖 Unit 3 - The Clinical Assistant (LLM)

The newest system in Future-Driven Diagnostics's portfolio is a language model that drafts clinical notes and answers staff questions. The evidence is no longer a lab value or a pixel - it's a **token**. The audit loop stays exactly the same: look at a prediction, explain it (now with attention weights and gradient-based token attribution instead of LIME/SHAP/Grad-CAM), stress-test with the same weight-randomization trick, and write a verdict.

We use a small, freely available pretrained model (`gpt2`) so this runs without a GPU.

In [ ]:
#@title 🔎 Pipeline diagram (double-click to view the code) { display-mode: "form" }
render_pipeline("Unit 3 workflow - LLM", "🤖", [
    ("📥", "Load weights", "small pretrained LLM"),
    ("💬", "Predict", "next-token candidates"),
    ("👁️", "Attention", "where does it look?"),
    ("🔥", "Gradient attr.", "which tokens drove it?"),
    ("🧪", "Stress-test", "randomize + re-explain"),
    ("✅", "Verdict", "audit sign-off")
])


## System 8 - Clinical assistant
**Model:** `gpt2`, pretrained, downloaded - not trained here (same "load weights" pattern as Unit 2).
**Claim under audit:** it completes clinical text sensibly. **Your question:** when it attends to a word, is it attending to something *clinically relevant*, or to punctuation / sentence position / the most recent word regardless of meaning?

In [ ]:
# ---- System 8: load gpt2 (small pretrained LLM, runs on CPU) ----
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
# attn_implementation="eager" so output_attentions returns real attention maps
try:
    llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME, attn_implementation="eager").to(DEVICE).eval()
except (TypeError, ValueError):
    llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME).to(DEVICE).eval()

prompt = "The patient presented with chest pain and shortness of breath, so the doctor ordered a"
inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
tokens_str = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    out = llm_model(**inputs)
probs = torch.softmax(out.logits[0, -1], dim=-1)
topk = torch.topk(probs, 5)
target_id = int(topk.indices[0])
print("Prompt:", prompt)
print("Top next-token candidates:")
for tid, p in zip(topk.indices.tolist(), topk.values.tolist()):
    print(f"  {tokenizer.decode([tid])!r:>15}   p={p:.3f}")

In [ ]:
def render_token_heatmap(tokens, weights, title=""):
    w = np.asarray(weights, dtype=float)
    w = (w - w.min()) / (np.ptp(w) + 1e-8)
    spans = ""
    for tok, wt in zip(tokens, w):
        tok_disp = tok.replace("Ġ", " ").replace("Ċ", "\\n")
        alpha = 0.15 + 0.75 * wt
        spans += (f'<span style="background:rgba(102,126,234,{alpha:.2f});padding:3px 6px;'
                  f'border-radius:6px;margin:2px;display:inline-block;font-family:ui-monospace,monospace;'
                  f'font-size:13px;color:#241d45">{tok_disp}</span>')
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;border:1px solid #ecebff;border-radius:14px;
            padding:14px 16px;background:#fbfaff;margin:8px 0">
  <div style="font-weight:800;color:#3b2d6b;margin-bottom:8px;font-size:14px">{title}</div>
  <div style="line-height:2.1">{spans}</div>
</div>'''))

### 🔍 Explain, method 1 - Attention
Attention is the model's own "where do I look?" signal. For the position where the next word is decided (the last token), we read the **last layer's attention**, averaged over heads, back across every prompt token. Bright tokens are the ones the model attended to most while forming its answer.

In [ ]:
# ---- attention: which prompt tokens does the decision point look at? ----
with torch.no_grad():
    att = llm_model(**inputs, output_attentions=True).attentions
# att[-1]: last layer [1, n_heads, seq, seq]; take attention FROM the last token, averaged over heads
attn_last = att[-1][0].mean(0)[-1]                       # [seq]
render_token_heatmap(tokens_str, attn_last.detach().cpu().numpy(),
                     "🔎 Attention from the decision point - which tokens the model looks at")
print("Bright = attended to. Is it the clinical words (chest pain, shortness of breath), or punctuation / the most recent token?")

### 🔍 Explain, method 2 - Gradient x embedding
Attention shows where the model *looked*; it doesn't prove those tokens *changed the answer*. Gradient x embedding does: take the gradient of the predicted next-token score with respect to each input-token embedding, dot it with the embedding, and take the magnitude. Bright tokens are the ones that actually pushed the prediction.

In [ ]:
# ---- gradient x embedding: which tokens DROVE the predicted next word? ----
def grad_x_emb(model, input_ids, attention_mask, target_id):
    """|grad . embedding| per input token: how much each token moved the target logit."""
    emb = model.get_input_embeddings()(input_ids).clone().detach().requires_grad_(True)
    logits = model(inputs_embeds=emb, attention_mask=attention_mask).logits
    model.zero_grad(set_to_none=True)
    logits[0, -1, target_id].backward()
    return (emb.grad[0] * emb[0]).sum(-1).abs().detach().cpu().numpy()   # [seq]

attr = grad_x_emb(llm_model, inputs["input_ids"], inputs["attention_mask"], target_id)
render_token_heatmap(tokens_str, attr,
                     "🔎 Gradient x embedding - which tokens drove the predicted next word")
print("Do the two maps agree? Attention can be diffuse where gradient attribution is sharp -- that disagreement is a finding.")

### 🧪 Stress-test
Randomize the weights and re-run **both** token maps on the same prompt. A faithful explanation of *this* model should change: attention should stop concentrating on the clinical words and the gradient map should scramble. If either barely moves, it was tracking sentence structure or position, not what the model learned.

In [ ]:
rnd_llm = randomize_weights(llm_model)

# attention on the randomized model
with torch.no_grad():
    attn_rnd = rnd_llm(**inputs, output_attentions=True).attentions[-1][0].mean(0)[-1]
render_token_heatmap(tokens_str, attn_rnd.detach().cpu().numpy(),
                     "🔎 Attention on the RANDOM model - compare to above")

# gradient x embedding on the randomized model (uses the random model's own embeddings)
attr_rnd = grad_x_emb(rnd_llm, inputs["input_ids"], inputs["attention_mask"], target_id)
render_token_heatmap(tokens_str, attr_rnd,
                     "🔎 Gradient x embedding on the RANDOM model - compare to above")
print("Whichever map changes MORE under randomisation is the one tracking the model, not the sentence.")

In [ ]:
#@title 📝 Verdict prompts (double-click to view the code) { display-mode: "form" }
verdict_box("Clinical assistant (LLM)", [
    "Do the attention map and the gradient-attribution map agree on which tokens matter?",
    "Did either explanation survive the stress-test better than the other?",
    "Try a different prompt -- does the model attend to clinically relevant words, or mostly to recent/positional tokens?",
    "Your verdict: trustworthy / trustworthy with caveats / not ready -- and one sentence why."
])


> **Your verdict - Clinical assistant (LLM)**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

---
# 🗂️ Final Audit Report

You've now run the same four-step audit - Look, Explain, Stress-test, Verdict - across eight systems in three completely different data modalities. Fill in your scorecard below.

In [ ]:
#@title 📊 Visualization - Final Scorecard (double-click to view the code) { display-mode: "form" }
def render_scorecard(rows):
    """rows: list of (icon, system, modality, verdict) tuples. Leave verdict blank
    ('') to render an empty box for students to reason about."""
    trs = ""
    colors = {"": "#e9e6f7", "Trustworthy": "#c9f2d8", "Caveats": "#fdeec7", "Not ready": "#fbd6d6"}
    for ic, sysname, modality, verdict in rows:
        bg = colors.get(verdict, "#e9e6f7")
        trs += (f'<tr><td style="padding:8px 10px;font-size:18px">{ic}</td>'
                f'<td style="padding:8px 10px;font-weight:700;color:#2c2350">{sysname}</td>'
                f'<td style="padding:8px 10px;color:#6b6485">{modality}</td>'
                f'<td style="padding:8px 10px"><span style="background:{bg};padding:4px 10px;'
                f'border-radius:8px;font-size:12px;font-weight:700;color:#3b2d6b">{verdict or "?"}</span></td></tr>')
    display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#fbf5ff);
            border-radius:18px;padding:18px 20px;margin:8px 0;border:1px solid #ecebff">
  <div style="font-weight:800;color:#3b2d6b;font-size:18px;margin-bottom:10px">🏥 Future-Driven Diagnostics Health -- AI Audit Scorecard</div>
  <table style="width:100%;border-collapse:collapse">{trs}</table>
</div>'''))

print("Edit the `verdict` string for each row below to 'Trustworthy', 'Caveats', or 'Not ready' and re-run that cell to update the scorecard.")

In [ ]:
render_scorecard([
    ("\U0001F4C8", "Cardiovascular risk score", "Tabular", ""),
    ("\U0001F3E5", "Readmission predictor", "Tabular", ""),
    ("\U0001F9E0", "Stroke triage flag", "Tabular", ""),
    ("\U0001FA7B", "Chest X-ray reader", "Imaging", ""),
    ("\U0001FA79", "Skin-lesion screener", "Imaging", ""),
    ("\U0001F397\uFE0F", "Breast-ultrasound screener", "Imaging", ""),
    ("\U0001F52C", "Colon-histopathology classifier", "Imaging", ""),
    ("\U0001F916", "Clinical assistant (LLM)", "Text", ""),
])

### Closing reflection
Write a short paragraph (5-8 sentences) answering:

1. Which single audit finding across all eight systems worried you most, and why?
2. Name one system where the **explanation method itself** could have misled you if you'd skipped the stress-test.
3. What's the one habit from this notebook you'd insist every team at Future-Driven Diagnostics adopt before deploying a new model?

> Remember the throughline: **strong metrics tell you a model is right often - not why, and not whether it's right for good reasons.** Attribution ≠ causation. A method that doesn't survive being stress-tested against a randomized model isn't explaining your model at all.